# Lab 5 — Structured Streaming sur les événements Wikimedia

Pipeline : `wikistream_producer.py` (hôte) → Kafka topic `wikistreams` → **ce notebook** (Spark Structured Streaming) → sink `console` (visible avec `docker logs -f pyspark_notebook`) + sink `memory` (interrogeable ici en SQL).

Plan :
1. Session Spark + lecture du topic Kafka
2. Parsing JSON avec un schéma enrichi + colonnes dérivées (`event_time`, `size_delta`…)
3. **Démo** : nombre d'éditions par robot/humain (fenêtre tumbling 1 h)
4. **Exploration A — fenêtres tumbling** (1 min) : volume et taille des éditions
5. **Exploration B — fenêtres glissantes / overlapping** (5 min, pas de 1 min) : utilisateurs les plus actifs
6. **Exploration C — fenêtres glissantes** (10 min, pas de 2 min) : pages les plus modifiées
7. **Exploration D — tumbling + filtre** (2 min) : grosses modifications (|delta| ≥ 1000 octets) par wiki
8. Visualisation dans le notebook (sink `memory`) puis arrêt des requêtes

In [ ]:
# Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, window, count, sum as _sum, avg, max as _max, min as _min,
    abs as _abs, when, approx_count_distinct, desc, round as _round, expr,
)
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, IntegerType, BooleanType,
)

: 

In [ ]:
# Session Spark avec le connecteur Kafka.
# Le connecteur doit avoir EXACTEMENT la version de Spark installée dans l'image
# (l'image jupyter/pyspark-notebook "latest" évolue : Spark 4.1, 4.2...).
import pyspark
kafka_package = f"org.apache.spark:spark-sql-kafka-0-10_2.13:{pyspark.__version__}"

spark = SparkSession.builder \
    .appName("WikistreamsStructuredStreaming") \
    .config("spark.jars.packages", kafka_package) \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

# Moins de logs INFO pour que les tables du sink console restent lisibles
spark.sparkContext.setLogLevel("WARN")
spark.version

## 1. Source Kafka

Depuis le conteneur on utilise le listener **INTERNAL** `kafka:19092` (et non `localhost:9092`, qui n'est valable que depuis l'hôte).

In [ ]:
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .load()

raw_df.printSchema()

## 2. Schéma enrichi et colonnes dérivées

Par rapport à la démo on ajoute `server_name`, `namespace` et `meta.dt`.

* `timestamp` est un epoch **en secondes** : on le *cast* directement en `TimestampType`.
  (`from_unixtime` de la démo renvoie une **chaîne**, ce qui fonctionne mais force une conversion implicite.)
* `size_delta = length.new - length.old` : octets ajoutés (>0) ou retirés (<0). Pour un événement `new`, `length.old` est nul → on prend 0.

In [ ]:
wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("server_name", StringType(), True),
    StructField("namespace", IntegerType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True),
    ]), True),
    StructField("meta", StructType([
        StructField("dt", StringType(), True),
        StructField("domain", StringType(), True),
    ]), True),
])

parsed_df = raw_df.select(
    from_json(col("value").cast("string"), wikimedia_schema).alias("data")
).select("data.*")

events_df = parsed_df \
    .withColumn("event_time", col("timestamp").cast("timestamp")) \
    .withColumn("old_len", when(col("length.old").isNull(), 0).otherwise(col("length.old"))) \
    .withColumn("new_len", when(col("length.new").isNull(), 0).otherwise(col("length.new"))) \
    .withColumn("size_delta", col("new_len") - col("old_len")) \
    .withColumn("abs_delta", _abs(col("size_delta"))) \
    .drop("length", "old_len", "new_len")

events_df.printSchema()

## 3. Démo du cours — éditions robot vs humain (tumbling 1 h)

Inchangée sur le fond : `window(event_time, "1 hour")` sans 3ᵉ argument = **fenêtre tumbling** (fenêtres contiguës, sans chevauchement, chaque événement appartient à exactement une fenêtre). Mode `complete` : toute la table de résultats est réécrite à chaque micro-batch.

In [ ]:
bot_counts_df = events_df \
    .groupBy(window(col("event_time"), "1 hour"), col("bot")) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "bot", "edit_count",
    )

q_demo = bot_counts_df.writeStream \
    .queryName("demo_bot_counts") \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .trigger(processingTime="30 seconds") \
    .start()

## 4. Exploration A — fenêtres **tumbling** de 1 minute : volume et taille des modifications

Pour chaque minute et chaque catégorie (bot / humain) :
nombre d'éditions, nombre d'éditeurs distincts (approx.), octets ajoutés, octets retirés, delta moyen, plus grosse modification.

**Watermark** de 2 minutes : Spark accepte les événements en retard jusqu'à 2 min après le plus grand `event_time` vu, puis *finalise* la fenêtre et purge son état → la mémoire ne grossit pas indéfiniment. Mode `update` : seules les lignes modifiées depuis le dernier batch sont affichées.

In [ ]:
tumbling_size_df = events_df \
    .withWatermark("event_time", "2 minutes") \
    .groupBy(window(col("event_time"), "1 minute"), col("bot")) \
    .agg(
        count("*").alias("edits"),
        approx_count_distinct("user").alias("editors"),
        _sum(when(col("size_delta") > 0, col("size_delta")).otherwise(0)).alias("bytes_added"),
        _sum(when(col("size_delta") < 0, -col("size_delta")).otherwise(0)).alias("bytes_removed"),
        _round(avg("size_delta"), 1).alias("avg_delta"),
        _max("abs_delta").alias("max_abs_delta"),
        _sum(when(col("minor"), 1).otherwise(0)).alias("minor_edits"),
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "bot", "edits", "editors", "bytes_added", "bytes_removed",
        "avg_delta", "max_abs_delta", "minor_edits",
    )

q_tumbling = tumbling_size_df.writeStream \
    .queryName("tumbling_1min_size") \
    .format("console") \
    .outputMode("update") \
    .option("truncate", False) \
    .trigger(processingTime="30 seconds") \
    .start()

## 5. Exploration B — fenêtres **glissantes (overlapping)** 5 min / pas de 1 min : utilisateurs les plus actifs

`window(event_time, "5 minutes", "1 minute")` : une nouvelle fenêtre de 5 min démarre **chaque minute**. Les fenêtres se chevauchent → **chaque événement est compté dans 5 fenêtres** (5 min / 1 min). C'est une moyenne mobile : la courbe est lissée par rapport au tumbling.

Mode `complete` pour pouvoir trier (`orderBy`) la table entière — le tri n'est autorisé qu'en mode `complete` sur un agrégat.

In [ ]:
sliding_users_df = events_df \
    .groupBy(
        window(col("event_time"), "5 minutes", "1 minute"),
        col("user"), col("bot"),
    ) \
    .agg(
        count("*").alias("edits"),
        _sum("size_delta").alias("net_bytes"),
        approx_count_distinct("title").alias("pages"),
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "user", "bot", "edits", "net_bytes", "pages",
    ) \
    .orderBy(desc("window_start"), desc("edits"))

q_sliding_users = sliding_users_df.writeStream \
    .queryName("sliding_5min_top_users") \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .option("numRows", 20) \
    .trigger(processingTime="30 seconds") \
    .start()

## 6. Exploration C — fenêtres **glissantes** 10 min / pas de 2 min : pages les plus modifiées

Même principe, chaque événement tombe dans 10/2 = 5 fenêtres. On cherche les pages « chaudes » (guerres d'édition, actualité…) et on garde le nombre d'éditeurs distincts pour distinguer *une* personne qui fait beaucoup de petites modifications d'un vrai sujet collectif.

Résultats envoyés dans un **sink `memory`** (table SQL `hot_pages`) pour les interroger depuis le notebook.

In [ ]:
hot_pages_df = events_df \
    .withWatermark("event_time", "5 minutes") \
    .groupBy(
        window(col("event_time"), "10 minutes", "2 minutes"),
        col("server_name"), col("title"),
    ) \
    .agg(
        count("*").alias("edits"),
        approx_count_distinct("user").alias("editors"),
        _sum("size_delta").alias("net_bytes"),
        _sum("abs_delta").alias("churn_bytes"),
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "server_name", "title", "edits", "editors", "net_bytes", "churn_bytes",
    )

q_hot_pages = hot_pages_df.writeStream \
    .queryName("hot_pages") \
    .format("memory") \
    .outputMode("complete") \
    .trigger(processingTime="30 seconds") \
    .start()

## 7. Exploration D — **tumbling** 2 min + filtre : grosses modifications par wiki

On filtre d'abord (`abs_delta >= 1000` octets, hors robots) puis on agrège par tranche de 2 minutes et par wiki. Utile avec un producteur lancé sur plusieurs langues, par ex. :

```bash
python wikistream_producer.py --wiki fr.wikipedia.org en.wikipedia.org de.wikipedia.org --type edit new
```

Mode `append` : une ligne n'est émise **qu'une seule fois**, quand le watermark a dépassé la fin de la fenêtre (résultat définitif). Il faut donc attendre ≈ 2 min (fenêtre) + 1 min (watermark) avant de voir les premières lignes.

In [ ]:
big_edits_df = events_df \
    .filter((col("abs_delta") >= 1000) & (col("bot") == False)) \
    .withWatermark("event_time", "1 minute") \
    .groupBy(window(col("event_time"), "2 minutes"), col("server_name"), col("type")) \
    .agg(
        count("*").alias("big_edits"),
        _sum(when(col("size_delta") > 0, 1).otherwise(0)).alias("additions"),
        _sum(when(col("size_delta") < 0, 1).otherwise(0)).alias("deletions"),
        _max("abs_delta").alias("largest"),
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "server_name", "type", "big_edits", "additions", "deletions", "largest",
    )

q_big_edits = big_edits_df.writeStream \
    .queryName("tumbling_2min_big_edits") \
    .format("console") \
    .outputMode("append") \
    .option("truncate", False) \
    .trigger(processingTime="30 seconds") \
    .start()

## 8. Suivi des requêtes et lecture du sink `memory`

Les sinks `console` écrivent dans les logs du conteneur :

```bash
docker logs -f pyspark_notebook
```

La table `hot_pages` (exploration C) peut être interrogée ici. Relancer la cellule pour voir l'évolution.

In [ ]:
for q in spark.streams.active:
    p = q.lastProgress
    print(f"{q.name:<28} actif={q.isActive}  "
          f"lignes_lues_dernier_batch={p['numInputRows'] if p else '-'}  "
          f"status={q.status['message']}")

In [ ]:
# Top 15 des pages chaudes sur les 10 dernières minutes.
# La fenêtre la plus récente (max window_start) ne contient que ~2 min de données ;
# celle qui couvre les 10 dernières minutes se termine juste après le dernier
# événement, soit max(window_end) - 8 min (10 min de fenêtre - 2 min de pas).
spark.sql("""
    SELECT window_start, window_end, server_name, title, edits, editors, net_bytes, churn_bytes
    FROM hot_pages
    WHERE window_end = (SELECT max(window_end) - INTERVAL 8 MINUTES FROM hot_pages)
    ORDER BY edits DESC, editors DESC
    LIMIT 15
""").show(truncate=False)

In [ ]:
# Comparaison tumbling vs glissant sur le même flux :
# une page modifiée N fois apparaît dans 5 fenêtres glissantes (10 min / 2 min),
# donc la somme des 'edits' sur toutes les fenêtres ≈ 5 × le nombre réel d'éditions.
spark.sql("""
    SELECT count(DISTINCT window_start) AS nb_fenetres,
           sum(edits)                   AS somme_edits_toutes_fenetres
    FROM hot_pages
""").show()

## 9. Arrêt propre

Arrêter toutes les requêtes avant de relancer des cellules (sinon deux requêtes avec le même `queryName` entrent en conflit).

In [ ]:
for q in spark.streams.active:
    print("stop", q.name)
    q.stop()